# 01 — NIFTY 50 data + realized variance estimators

Walks through:
1. Download NIFTY 50 OHLCV from yfinance
2. Compute realized variance using range-based estimators
3. Visualize the volatility series
4. Empirical stylized facts (volatility clustering, leverage, fat tails)

**Prerequisite:** `pip install -r requirements.txt`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loaders import download_nifty_yfinance
from src.data.realized_variance import add_realized_variance_columns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Download NIFTY 50 daily OHLCV

In [ ]:
df = download_nifty_yfinance(
    start='2007-01-01',
    save_path='../data/processed/nifty.parquet',
)
print(f'{len(df):,} trading days from {df.index[0].date()} to {df.index[-1].date()}')
df.tail()

## 2. Compute realized variance estimators

In [ ]:
df = add_realized_variance_columns(df)
df = df.dropna(subset=['rv_yang_zhang', 'log_return'])
print(f'After dropping NaNs: {len(df):,} days')
df[['Close', 'log_return', 'rv_yang_zhang', 'rv_yz_annualized']].tail()

## 3. NIFTY 50 price and annualized realized volatility through time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['Close'], color='steelblue', linewidth=0.7)
axes[0].set_ylabel('NIFTY 50 Close')
axes[0].set_title('NIFTY 50 — price and realized volatility')

axes[1].plot(df.index, df['rv_yz_annualized'] * 100, color='crimson', linewidth=0.5)
axes[1].set_ylabel('Annualized RV (%)')
axes[1].set_xlabel('Date')
axes[1].axhline(20, color='gray', linestyle='--', alpha=0.5, label='20% reference')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Stylized fact: volatility clustering

ACF of squared returns should show significant positive autocorrelation at many lags.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df['log_return'].dropna(), lags=40, ax=axes[0], title='ACF of log returns')
plot_acf(df['log_return'].dropna() ** 2, lags=40, ax=axes[1], title='ACF of squared returns (vol clustering)')
plt.tight_layout()
plt.show()

## 5. Stylized fact: fat tails

In [ ]:
from scipy import stats

ret = df['log_return'].dropna()
kurt = stats.kurtosis(ret)
skew = stats.skew(ret)

print(f'Skewness: {skew:.3f}')
print(f'Excess kurtosis: {kurt:.3f} (Gaussian = 0; > 0 means fatter tails)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ret * 100, bins=120, density=True, alpha=0.7, color='steelblue', label='NIFTY daily returns')
x = np.linspace(ret.min() * 100, ret.max() * 100, 400)
ax.plot(x, stats.norm.pdf(x, ret.mean() * 100, ret.std() * 100), 'r--', label='Normal fit')
ax.set_xlabel('Daily log return (%)')
ax.set_ylabel('Density')
ax.set_title('Return distribution vs Gaussian (note the fat tails)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 6. Stylized fact: leverage effect (negative correlation between returns and future volatility)

In [ ]:
leverage_corr = df['log_return'].corr(df['rv_yang_zhang'].shift(-1))
print(f'Corr(return_t, RV_{{t+1}}) = {leverage_corr:.4f}')
print('(Negative value confirms the leverage effect — bad return days are followed by elevated volatility)')

## Next steps

→ `02_baselines.ipynb`: Fit GARCH(1,1), EGARCH, HAR-RV models and compare forecasts.

Or from the command line:
```bash
python -m src.training.train_all --models har garch xgb --initial-train-size 1000
```